# Signal Persistence: Redis → InfluxDB

This notebook demonstrates the signal persistence pipeline:

1. Subscribe to `market:TradeSignal:*` via RedisSubscription (queue mode)
2. Receive TradeSignal events in real-time
3. Persist each to InfluxDB via `TelegrafHTTPEventProcessor.process_event()`
4. Query back from InfluxDB to verify persistence

This is the notebook equivalent of `tasty-signal persist` — the same
EngineRunner wiring, but using queue-based consumption for interactive use.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import asyncio
import logging

import influxdb_client

from tastytrade.common.logging import setup_logging
from tastytrade.config import RedisConfigManager
from tastytrade.connections import InfluxCredentials
from tastytrade.providers.subscriptions import RedisSubscription
from tastytrade.messaging.processors.influxdb import TelegrafHTTPEventProcessor
from tastytrade.analytics.engines.models import TradeSignal

In [ ]:
logging.getLogger().handlers.clear()
setup_logging(level=logging.INFO, console=True, file=False)

## Connect to Redis and InfluxDB

In [ ]:
config = RedisConfigManager()
config.initialize()

subscription = RedisSubscription(config=config)
await subscription.connect()

processor = TelegrafHTTPEventProcessor()
print("Connected to Redis and InfluxDB")

## Subscribe to TradeSignal channels

Uses queue mode (no `on_update` callback) for interactive consumption.
In production, the EngineRunner wires `on_update=processor.process_event` directly.

In [ ]:
channel = "market:TradeSignal:hull_macd:SPX*"
await subscription.subscribe(channel, event_type=TradeSignal)
print(f"Subscribed to {channel}")

## Consume and persist signals

Wait for TradeSignal events on the queue and persist each one to InfluxDB.
Run the signal detection service (`tasty-signal run --symbols SPX --intervals 5m`)
in a separate terminal to generate signals.

The loop processes up to `max_signals` events or exits on timeout — no
unbounded `while True`. Every loop must declare its termination condition.

In [ ]:
queue_key = "TradeSignal:SPX{=5m}"
max_signals = 20
timeout_seconds = 60.0
persisted_count = 0

try:
    while persisted_count < max_signals:
        event = await asyncio.wait_for(
            subscription.queue[queue_key].get(), timeout=timeout_seconds
        )
        processor.process_event(event)
        persisted_count += 1
        print(
            f"Persisted #{persisted_count}/{max_signals}: "
            f"{event.signal_type} {event.direction} {event.eventSymbol} "
            f"at {event.start_time} (trigger={event.trigger})"
        )
except asyncio.TimeoutError:
    print(f"\nTimeout — no signals received in {timeout_seconds}s. Persisted {persisted_count} total.")

print(f"\nDone. Persisted {persisted_count} signals.")

## Verify: Query signals from InfluxDB

Read back persisted TradeSignal records to confirm InfluxDB writes.

In [ ]:
from tastytrade.providers.market import MarketDataProvider
from datetime import datetime, timedelta, timezone

influx_creds = InfluxCredentials(config=config)
influxdb = influxdb_client.InfluxDBClient(
    url=influx_creds.url,
    token=influx_creds.token,
    org=influx_creds.org,
)
streamer = MarketDataProvider(subscription, influxdb)

now = datetime.now(timezone.utc)
signals = streamer.download_signals(
    symbol="SPX{=5m}",
    start=now - timedelta(days=1),
    stop=now,
)
print(f"Retrieved {len(signals)} signals from InfluxDB")
for s in signals:
    print(f"  {s.signal_type} {s.direction} at {s.start_time} (trigger={s.trigger})")

## Cleanup

In [ ]:
processor.close()
await subscription.close()
print("Connections closed")